# Пространственные отношения

В предыдущем разделе мы изменяли геометрию объектов. Во многих задачах важно и другое: **как объекты расположены относительно друг друга**. Пересекаются ли они? Находится ли один внутри другого? Касаются ли их границы?

На этих вопросах основаны многие задачи пространственного анализа:

- поиск объектов в заданной зоне;
- фильтрация данных по местоположению;
- пространственное объединение данных из разных слоёв (следующий раздел).

В GeoPandas для этого используются **пространственные предикаты** (_spatial predicates_): `within`, `contains`, `intersects` и другие.


## 0. Импорт библиотек и подготовка данных

### 0.1 Импорт библиотек


In [ ]:
import pandas as pd
import geopandas as gpd

### 0.2 Подготовка данных

В этом разделе используются файлы из папки [`data`](https://github.com/bella-mir/spatialAnalysisFall26/tree/main/notebooks/data) нашего репозитория:

- **spb_admin.gpkg** — границы районов (слой `district`) и муниципальных округов (слой `okrug`) Санкт-Петербурга.
  Источник: материалы курса «Методы пространственного анализа», НИУ ВШЭ (Р. Гончаров)
- **spb_theaters.csv** — театры Санкт-Петербурга.
  Источник: [Портал открытых данных Санкт-Петербурга](https://data.gov.spb.ru/)

Прочитаем границы районов и выберем Центральный район:


In [ ]:
districts = gpd.read_file("../data/spb_admin.gpkg", layer="district")

central = districts[districts["NAME"] == "Центральный район"]
central_geom = central.geometry.iloc[0]

central.explore()

Прочитаем данные о театрах и создадим на их основе `GeoDataFrame`:


In [ ]:
theaters_csv = pd.read_csv("../data/spb_theaters.csv")
theaters_csv = theaters_csv.dropna(subset=["longitude", "latitude"])

theaters = gpd.GeoDataFrame(
    theaters_csv,
    geometry=gpd.points_from_xy(theaters_csv["longitude"], theaters_csv["latitude"]),
    crs=4326
)

theaters.explore()

## 1. Топологические отношения

**Топологические** (пространственные) отношения описывают взаимное расположение двух геометрий. Они отвечают на вопросы «пересекаются ли объекты?», «находится ли один объект внутри другого?», «касаются ли их границы?».

### 1.1 Основные отношения

| **Отношение**                   | **Условие истинности**                                                                                                              |
| ------------------------------- | ----------------------------------------------------------------------------------------------------------------------------------- |
| **intersects** (_пересекается_) | У геометрий есть хотя бы одна общая точка — на границе или внутри. Противоположно `disjoint`.                                       |
| **disjoint** (_раздельны_)      | У геометрий нет ни одной общей точки.                                                                                               |
| **within** (_внутри_)           | Геометрия A полностью находится внутри геометрии B. Эквивалентно `B.contains(A)`.                                                   |
| **contains** (_содержит_)       | Геометрия A полностью содержит геометрию B. Эквивалентно `B.within(A)`.                                                             |
| **touches** (_касается_)        | Геометрии имеют общие точки только на границе, их внутренние области не пересекаются.                                               |
| **overlaps** (_перекрывается_)  | Геометрии одной размерности частично перекрываются, но ни одна не содержит другую полностью.                                        |
| **crosses** (_пересекает_)      | Геометрии пересекаются внутренними точками, а результат пересечения имеет меньшую размерность (например, линия пересекает полигон). |


### 1.2 Отношения в GeoPandas

В GeoPandas отношения реализованы как методы, которые проверяют условие для каждой геометрии и возвращают `True` или `False`.

Проверим, какие театры находятся внутри Центрального района. Театры и районы хранятся в одной системе координат (EPSG:4326), поэтому перепроецирование не требуется.


In [ ]:
theaters.within(central_geom)

## 2. Фильтрация по пространственному признаку

Полученную серию `True`/`False` можно использовать как фильтр — так же, как обычное логическое условие в pandas. Отберём театры, которые находятся в Центральном районе:


In [ ]:
theaters_center = theaters[theaters.within(central_geom)]

print(f"Театров в Центральном районе: {len(theaters_center)} из {len(theaters)}")

m = central.explore(color="gray", style_kwds={"fillOpacity": 0.1})
theaters_center.explore(m=m, color="red")

## 3. Выбор предиката

Для точек выбор предиката обычно очевиден: точка либо внутри полигона, либо нет. С полигонами сложнее. Попробуем найти муниципальные округа, которые входят в Центральный район:


In [ ]:
okrugs = gpd.read_file("../data/spb_admin.gpkg", layer="okrug")

okrugs_within = okrugs[okrugs.within(central_geom)]
okrugs_intersects = okrugs[okrugs.intersects(central_geom)]

print("within:    ", len(okrugs_within))
print("intersects:", len(okrugs_intersects))

In [ ]:
okrugs_within

Оба результата неверны:

- `within` нашёл слишком мало округов. Границы округов и районов оцифрованы с небольшими расхождениями, поэтому край округа может на доли метра выходить за границу района, и условие «полностью внутри» не выполняется.
- `intersects` нашёл слишком много: в результат попали все соседние округа, у которых есть хотя бы одна общая точка с районом.

Посмотрим на карте:


In [ ]:
m = okrugs_intersects.explore(color="orange", style_kwds={"fillOpacity": 0.3})
central.boundary.explore(m=m, color="black")

Надёжный способ — проверять не весь полигон, а **одну его внутреннюю точку**. Метод `representative_point()` возвращает точку, которая гарантированно лежит внутри полигона:


In [ ]:
okrugs_center = okrugs[okrugs.representative_point().within(central_geom)]

print("representative_point + within:", len(okrugs_center))
okrugs_center[["NAME"]]

In [ ]:
m = okrugs_center.explore(column="NAME", legend=False)
central.boundary.explore(m=m, color="black")

> Расхождения границ — обычная ситуация, когда слои получены из разных источников или оцифрованы в разное время. Прежде чем использовать строгие предикаты (`within`, `contains`, `touches`) для полигонов, проверьте результат на карте.


## Итог

В этом разделе мы познакомились с **топологическими отношениями** и научились применять их для анализа данных:

- узнали, какие бывают пространственные отношения;
- использовали методы GeoPandas для проверки отношений и фильтрации объектов по местоположению;
- увидели, что для полигонов выбор предиката влияет на результат, и научились обходить проблему расхождения границ с помощью `representative_point()`.

Мы использовали отношения для фильтрации. Но часто нужно не только отобрать объекты, но и объединить атрибуты двух слоёв. Для этого используется **пространственное объединение**, которое мы рассмотрим в следующем разделе.
